In [1]:
import sys
import os
import json
import urllib3
import requests
from configparser import ConfigParser

# Add your local ThreatConnect SDK to path
sys.path.append("Z:/HTOC/Data_Analytics/threatconnect")
from ThreatConnect import ThreatConnect
from RequestObject import RequestObject
from Owners import Owners

# Add your project repo to path
project_root = r"C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull"
if project_root not in sys.path:
    sys.path.append(project_root)

from utils.config_loader import load_config

# Load API config
config_path = os.path.join(project_root, "utils", "config.json")
try:
    api_secret_key, api_access_id, api_base_url, api_default_org = load_config(config_path)
    print(f"Loaded config from: {config_path}")
    print(f"Base URL: {api_base_url}")
    print(f"Access ID: {api_access_id}")
    print(f"Default Org: {api_default_org}")
except Exception as e:
    print(f"[ERROR] Failed to load configuration: {e}")
    sys.exit(1)

# Disable SSL verification warnings (use cautiously)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
verify_ssl = False

# Initialize ThreatConnect session
try:
    tc = ThreatConnect(api_access_id, api_secret_key, api_default_org, api_base_url)
    print("ThreatConnect initialized.")
except Exception as e:
    print(f"[ERROR] Failed to initialize ThreatConnect: {e}")
    sys.exit(1)

# Define the owner (organization scope)
owner = 'HTOC Org'

# Create a request object to fetch indicators (or other data)
try:
    ro = RequestObject()
    ro.set_http_method('GET')
    ro.set_owner(owner)
    ro.set_owner_allowed(True)
    # ro.set_resource_pagination(True)  # Uncomment if needed
    print("RequestObject successfully created.")
except Exception as e:
    print(f"[ERROR] Failed to initialize RequestObject: {e}")
    sys.exit(1)


Loaded config from: C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull\utils\config.json
Base URL: https://hvs.threatconnect.com/api
Access ID: 09783848890162390382
Default Org: HTOC Org
ThreatConnect initialized.
RequestObject successfully created.


In [2]:
import pandas as pd
from datetime import datetime

# Define time period
start = "2025-01-01T00:00:00Z"  # adjust as needed

# List of owners to pull from
list_of_owners = ['HTOC Org', 'HTOC Comm', 'HTOC legacy data']
final_results = []

for owner in list_of_owners:
    print(f"\nQuerying owner: {owner}")
    try:
        # Build TQL query string
        tql = f'ownerName EQ "{owner}" and lastObserved >= "{start}" and (indicatorActive EQ true OR indicatorActive EQ false)'

        # Set up the request
        ro.set_http_method('GET')
        ro.set_request_uri(f'/v3/indicators?tql={tql}&fields=threatAssess,associatedGroups,associatedIndicators,tags,observations&resultStart=0&resultLimit=10000')

        # Send the request
        response = tc.api_request(ro)

        # Parse response
        if response.headers.get('content-type') == 'application/json':
            results = response.json()
            final_results.append(results)
        else:
            print(f"Unexpected response format: {response.headers.get('content-type')}")

    except Exception as e:
        print(f"Failed to query indicators for {owner}: {e}")

# Normalize and clean results
if final_results:
    observed_src = pd.json_normalize(final_results, record_path='data')
    observed_src['indicator'] = observed_src['summary'].str.split(' ', expand=True)[0].str.strip()
    print(f"\nRetrieved {len(observed_src)} observed indicators.")
else:
    print("\nNo indicators retrieved.")



Querying owner: HTOC Org


Exiting: HTTPSConnectionPool(host='hvs.threatconnect.com', port=443): Max retries exceeded with url: /api/v3/indicators?tql=ownerName%20EQ%20%22HTOC%20Org%22%20and%20lastObserved%20%3E=%20%222025-01-01T00:00:00Z%22%20and%20(indicatorActive%20EQ%20true%20OR%20indicatorActive%20EQ%20false)&fields=threatAssess,associatedGroups,associatedIndicators,tags,observations&resultStart=0&resultLimit=10000&createActivityLog=false&owner=HTOC+Org (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x0000000019B14AD0>: Failed to resolve 'hvs.threatconnect.com' ([Errno 11001] getaddrinfo failed)"))


Failed to query indicators for HTOC Org: HTTPSConnectionPool(host='hvs.threatconnect.com', port=443): Max retries exceeded with url: /api/v3/indicators?tql=ownerName%20EQ%20%22HTOC%20Org%22%20and%20lastObserved%20%3E=%20%222025-01-01T00:00:00Z%22%20and%20(indicatorActive%20EQ%20true%20OR%20indicatorActive%20EQ%20false)&fields=threatAssess,associatedGroups,associatedIndicators,tags,observations&resultStart=0&resultLimit=10000&createActivityLog=false&owner=HTOC+Org (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x0000000019B14AD0>: Failed to resolve 'hvs.threatconnect.com' ([Errno 11001] getaddrinfo failed)"))

Querying owner: HTOC Comm


Exiting: HTTPSConnectionPool(host='hvs.threatconnect.com', port=443): Max retries exceeded with url: /api/v3/indicators?tql=ownerName%20EQ%20%22HTOC%20Comm%22%20and%20lastObserved%20%3E=%20%222025-01-01T00:00:00Z%22%20and%20(indicatorActive%20EQ%20true%20OR%20indicatorActive%20EQ%20false)&fields=threatAssess,associatedGroups,associatedIndicators,tags,observations&resultStart=0&resultLimit=10000&createActivityLog=false&owner=HTOC+Org (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x0000000019B175D0>: Failed to resolve 'hvs.threatconnect.com' ([Errno 11001] getaddrinfo failed)"))


Failed to query indicators for HTOC Comm: HTTPSConnectionPool(host='hvs.threatconnect.com', port=443): Max retries exceeded with url: /api/v3/indicators?tql=ownerName%20EQ%20%22HTOC%20Comm%22%20and%20lastObserved%20%3E=%20%222025-01-01T00:00:00Z%22%20and%20(indicatorActive%20EQ%20true%20OR%20indicatorActive%20EQ%20false)&fields=threatAssess,associatedGroups,associatedIndicators,tags,observations&resultStart=0&resultLimit=10000&createActivityLog=false&owner=HTOC+Org (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x0000000019B175D0>: Failed to resolve 'hvs.threatconnect.com' ([Errno 11001] getaddrinfo failed)"))

Querying owner: HTOC legacy data


Exiting: HTTPSConnectionPool(host='hvs.threatconnect.com', port=443): Max retries exceeded with url: /api/v3/indicators?tql=ownerName%20EQ%20%22HTOC%20legacy%20data%22%20and%20lastObserved%20%3E=%20%222025-01-01T00:00:00Z%22%20and%20(indicatorActive%20EQ%20true%20OR%20indicatorActive%20EQ%20false)&fields=threatAssess,associatedGroups,associatedIndicators,tags,observations&resultStart=0&resultLimit=10000&createActivityLog=false&owner=HTOC+Org (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x0000000019B22510>: Failed to resolve 'hvs.threatconnect.com' ([Errno 11001] getaddrinfo failed)"))


Failed to query indicators for HTOC legacy data: HTTPSConnectionPool(host='hvs.threatconnect.com', port=443): Max retries exceeded with url: /api/v3/indicators?tql=ownerName%20EQ%20%22HTOC%20legacy%20data%22%20and%20lastObserved%20%3E=%20%222025-01-01T00:00:00Z%22%20and%20(indicatorActive%20EQ%20true%20OR%20indicatorActive%20EQ%20false)&fields=threatAssess,associatedGroups,associatedIndicators,tags,observations&resultStart=0&resultLimit=10000&createActivityLog=false&owner=HTOC+Org (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x0000000019B22510>: Failed to resolve 'hvs.threatconnect.com' ([Errno 11001] getaddrinfo failed)"))

No indicators retrieved.


In [ ]:
import pandas as pd

# Step 1: Work on a copy
df = observed_src.copy()

# Step 2: Unpack tags.data
if 'tags.data' in df.columns:
    tags_expanded = df.explode('tags.data').reset_index(drop=True)
    tags_expanded = pd.concat(
        [tags_expanded.drop(columns=['tags.data']),
         tags_expanded['tags.data'].apply(pd.Series)], axis=1
    )
    tags_expanded = tags_expanded.add_prefix('tag_')
    tags_expanded.rename(columns={'tag_indicator': 'indicator'}, inplace=True)

# Step 3: Unpack associatedGroups.data
if 'associatedGroups.data' in df.columns:
    groups_expanded = df.explode('associatedGroups.data').reset_index(drop=True)
    groups_expanded = pd.concat(
        [groups_expanded.drop(columns=['associatedGroups.data']),
         groups_expanded['associatedGroups.data'].apply(pd.Series)], axis=1
    )
    groups_expanded = groups_expanded.add_prefix('group_')
    groups_expanded.rename(columns={'group_indicator': 'indicator'}, inplace=True)

# Step 4: Merge expanded data properly
if 'tags_expanded' in locals() and 'groups_expanded' in locals():
    observed_src = pd.merge(tags_expanded, groups_expanded, on='indicator', how='outer')
elif 'tags_expanded' in locals():
    observed_src = tags_expanded
elif 'groups_expanded' in locals():
    observed_src = groups_expanded

# Step 5: Display result
observed_src.head()


,tag_id,tag_dateAdded,tag_ownerId,tag_ownerName,tag_webLink,tag_type,tag_lastModified,tag_rating,tag_confidence,tag_threatAssessRating,...,group_documentType,group_documentDateAdded,group_publishDate,group_0,group_fileName,group_fileSize,group_escalated,group_reminded,group_overdue,group_assignments
0,5629499535004222,2025-02-11T16:19:08Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-04-24T07:35:33Z,5.0,90.0,5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,5629499535004222,2025-02-11T16:19:08Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-04-24T07:35:33Z,5.0,90.0,5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,5629499535004222,2025-02-11T16:19:08Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-04-24T07:35:33Z,5.0,90.0,5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,5629499535004222,2025-02-11T16:19:08Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-04-24T07:35:33Z,5.0,90.0,5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5629499535004222,2025-02-11T16:19:08Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-04-24T07:35:33Z,5.0,90.0,5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
